# Stochastic Simulation — Exercises & Methods

## A. Bayesian Posterior Derivations

**When**: Derive the posterior distribution given prior and likelihood.

### Recipe — Gaussian Conjugacy (1D)
1. Write p(x|y) ∝ p(y|x)p(x)
2. Expand exponential: collect all terms involving x
3. Group as −½(ax² − 2bx + c) where a = 1/σ² + 1/σ₀²
4. Complete the square: a(x − b/a)² + (c − b²/a)
5. Read off: μ_p = b/a, σ_p² = 1/a

### Recipe — Multivariate Gaussian
1. Write p(x|y) ∝ exp{−½[x^TΛx − 2x^Tt + const]} where Λ = A^TΣ⁻¹A + Σ₀⁻¹, t = A^TΣ⁻¹y + Σ₀⁻¹μ₀
2. Complete the square: (x − Λ⁻¹t)^TΛ(x − Λ⁻¹t)
3. Posterior: N(Λ⁻¹t, Λ⁻¹)

### Recipe — Non-Gaussian Conjugates
- Poisson-Gamma: p(x|y) ∝ x^{α−1+y}e^{−(β+1)x} → Gamma(α+y, β+1)
- Pattern: recognise kernel of known distribution

### Common Exam Tasks
- Derive 1D Gaussian posterior with multiple observations: σ_p² = (n/σ² + 1/σ₀²)⁻¹, μ_p = σ_p²(nȳ/σ² + μ₀/σ₀²)
- Derive marginal likelihood p(y) by integrating out x
- Model comparison: compute p₀(y) vs p₁(y), larger wins

## B. Inverse Transform Sampling

**When**: p* has a CDF with closed-form inverse.

### Recipe
1. Compute CDF: F(x) = ∫_{−∞}^x p*(t)dt
2. Invert: solve U = F(x) for x = F⁻¹(U)
3. Algorithm: U ~ Unif(0,1), X = F⁻¹(U)

### Discrete Version
1. Compute cumulative weights: F(s_k) = Σ_{j=1}^k w_j
2. Draw U ~ Unif(0,1)
3. Find smallest k such that F(s_k) ≥ U

### Key Derivations to Know

| Target | F(x) | F⁻¹(u) |
|--------|-------|--------|
| Exp(λ) | 1−e^{−λx} | −ln(1−u)/λ |
| Cauchy | ½ + arctan(x)/π | tan(π(u−½)) |
| Weibull(k) | 1−e^{−x^k} | (−ln(1−u))^{1/k} |
| Unif(a,b) | (x−a)/(b−a) | a + u(b−a) |

### When Inverse Transform Fails
- Gaussian: CDF has no closed-form inverse → use Box-Müller instead
- Complex distributions → use rejection sampling

## C. Transformation of Random Variables

**When**: Prove that Y = g(X) has a specific distribution.

### Recipe — 1D
1. Given X ~ p_X, Y = g(X)
2. Find inverse: x = g⁻¹(y)
3. Compute derivative: |dg⁻¹/dy|
4. Apply formula: p_Y(y) = p_X(g⁻¹(y)) |dg⁻¹/dy|
5. Recognise the resulting density

### Recipe — 2D (and higher)
1. Given (X₁,X₂) ~ p, (Y₁,Y₂) = g(X₁,X₂)
2. Find inverse: (x₁,x₂) = g⁻¹(y₁,y₂)
3. Compute Jacobian matrix J_{g⁻¹} (matrix of partial derivatives)
4. Compute |det J_{g⁻¹}|
5. Apply: p_{Y₁,Y₂} = p_{X₁,X₂}(g⁻¹(y)) |det J_{g⁻¹}|

### Box-Müller Proof Walkthrough
- X₁ ~ Exp(1/2), X₂ ~ Unif(−π,π)
- g: Y₁ = √X₁ cos X₂, Y₂ = √X₁ sin X₂
- g⁻¹: x₁ = y₁²+y₂², x₂ = arctan(y₂/y₁)
- J = [[2y₁, 2y₂], [−y₂/(y₁²+y₂²), y₁/(y₁²+y₂²)]]
- |det J| = 2
- p_{Y₁,Y₂} = (1/2)e^{−(y₁²+y₂²)/2} × (1/2π) × 2 = N(y₁;0,1)N(y₂;0,1) ✓

### Disk Sampling (Example 2.6)
- r ~ Unif(0,1), θ ~ Unif(−π,π), set X₁ = √r cosθ, X₂ = √r sinθ
- Same Jacobian |det J| = 2
- Result: p(x₁,x₂) = Unif(x₁²+x₂²; 0,1) × (1/2π) × 2 = 1/π for x₁²+x₂² < 1
- **Warning**: without √r, you get p ∝ 1/√(x₁²+x₂²) — NOT uniform!

## D. Rejection Sampling Design

**When**: Sample from p*(x) (or p̄*(x)) using proposal q(x).

### Recipe — Basic Rejection Sampler
1. Choose proposal q(x) that is easy to sample from
2. Find M such that p̄*(x) ≤ Mq(x) for all x:
   - Compute R(x) = p̄*(x)/q(x)
   - Find x* = argmax R(x): take log, differentiate, set to zero, check 2nd derivative
   - M = R(x*)
3. Run Algorithm 5/6: sample X'~q, U~Unif, accept if U ≤ p̄*(X')/(Mq(X'))
4. Acceptance rate: â = 1/M (normalised) or Z/M (unnormalised)

### Recipe — Optimising the Proposal
1. Parameterise q_θ(x)
2. For each θ: compute M_θ = sup_x p̄*(x)/q_θ(x)
3. Minimise: θ* = argmin_θ log M_θ
4. Take derivative of log M w.r.t. θ, set to zero

### Example Walkthrough: Gamma(α,1) target, Exp(λ) proposal
1. R(x) = x^{α−1}e^{(λ−1)x} / (λΓ(α))
2. log R: (α−1)log x + (λ−1)x − log λ − log Γ(α)
3. d/dx log R = (α−1)/x + (λ−1) = 0 → x* = (α−1)/(1−λ)
4. Compute M(λ) from x*
5. Minimise log M w.r.t. λ → λ* = 1/α
6. Final M* = e^{−(α−1)}/Γ(α)

### Checklist for Rejection Problems
- [ ] Is p̄* given or p*? (unnormalised vs normalised)
- [ ] Does Mq(x) ≥ p̄*(x) for ALL x? (check boundaries too)
- [ ] Is M the smallest possible? (optimised x*)
- [ ] Can you further optimise q by tuning parameters?

## E. Rejection Sampling for Bayesian Posteriors

**When**: Sample from p(x|y) without deriving the closed-form posterior.

### Recipe
1. Unnormalised posterior: p̄(x|y) = p(y|x)p(x)
2. Choose proposal q(x) (e.g. the prior, or a Gaussian)
3. Find M: sup_x p(y|x)p(x) / q(x)
4. Rejection algorithm with p̄(x|y) / (Mq(x))

### Example 2.13 — Gaussian Posterior via Rejection
- Prior N(0,1), likelihood N(y;x,σ²), observe y
- p̄(x|y) = N(y;x,σ²)N(x;0,1)
- Proposal: q(x) = N(x;0,1) (the prior itself!), M = 1
- Accept X' if U ≤ N(y;X',σ²) / sup_x N(y;x,σ²)

### Example 2.16 — Poisson-Gamma Posterior via Rejection
- p̄(x|y) = x^{α−1+y}e^{−2x}
- Proposal: Exp(λ), optimise λ* = 2/(α+y)
- Find x*, compute M, run rejection

### Key Insight
Rejection sampling with unnormalised density: don't need to know Z = p(y)!

## F. Composition & Marginalisation

**When**: Sample from joint, marginal, or mixture distributions.

### Joint Sampling Recipe
1. Factorise: p(x₁,…,x_n) = p(x₁)p(x₂|x₁)…p(x_n|x₁,…,x_{n−1})
2. Sample sequentially: X₁ ~ p(x₁), then X₂|X₁, etc.

### Marginal Sampling Recipe
1. Sample from joint: (X₁,X₂) ~ p(x₁,x₂)
2. Keep only X₁ → automatically distributed as p(x₁) = ∫p(x₁,x₂)dx₂

### Mixture Sampling Recipe
1. p*(x) = Σ_k w_k q_k(x)
2. Sample component: k ~ Categorical(w₁,…,w_K)
3. Sample value: X ~ q_k(x)

### Beta from Gamma (Exercise 2.8)
If X₁ ~ Gamma(α,1), X₂ ~ Gamma(β,1), then Y = X₁/(X₁+X₂) ~ Beta(α,β)

## G2. Importance Sampling Design

**When**: Estimate E_{p*}[φ(X)] or normalising constant Z when direct sampling from p* is hard.

### Recipe — Basic IS
1. Choose proposal q(x) covering the support of p*
2. Sample X^(i) ~ q, i=1,…,N
3. Compute weights: W^(i) = p̄*(X^(i))/q(X^(i))
4. IS estimate of Z: Ẑ = (1/N)Σ W^(i)
5. SNIS estimate: φ̂ = Σ w̄^(i)φ(X^(i)), where w̄^(i) = W^(i)/Σ W^(j)

### Recipe — Checking IS Quality
1. Compute ESS = (Σ W^(i))² / Σ(W^(i))²
2. If ESS/N is low (< 0.5): proposal is poor — redesign q
3. Inspect weight histogram: should not have one dominant weight

### Recipe — SIR (Unweighted Samples)
1. Run IS as above to get {X^(i), w̄^(i)}
2. Resample: draw N indices j₁,…,j_N ~ Categorical(w̄^(1),…,w̄^(N))
3. Output: {X^(j_k)} are approximately i.i.d. from p*

### Recipe — Log-Weight Implementation
1. log W^(i) = log p̄*(X^(i)) − log q(X^(i))
2. L = max_i log W^(i)
3. w̄^(i) = exp(log W^(i) − L) / Σ exp(log W^(j) − L)

### Common Exam Tasks
- Derive IS weights for a specific target/proposal pair
- Compute ESS from given weights
- Show IS estimator is unbiased (E[W·φ] = Z·E_{p*}[φ])
- Prove optimal proposal is q* ∝ |φ|p* (Prop 3.5)
- Bayesian IS with prior as proposal: W^(i) = p(y|X^(i))

## H2. Designing MH Samplers

**When**: Sample from p*(x) using MCMC; know p̄*(x) up to normalisation.

### Recipe — Standard MH
1. Choose proposal q(x'|x) (common: random walk N(x, σ²I), independent N(μ,Σ))
2. Initialise x₀; choose burn-in B and total iterations T
3. For t=1,…,T: propose x' ~ q(·|x_{t-1})
4. Compute log α = log p̄*(x') + log q(x_{t-1}|x') − log p̄*(x_{t-1}) − log q(x'|x_{t-1})
5. Accept if log U ≤ log α
6. Discard first B samples (burn-in)

### Recipe — Random Walk MH (symmetric q)
- If q(x'|x) = q(x|x') (e.g. N(x, σ²I)): log α = log p̄*(x') − log p̄*(x)
- Tune σ: too small → high acceptance but slow exploration; too large → low acceptance
- Rule of thumb: target ~23% acceptance rate in high-d, ~44% in 1D

### Recipe — Proving Detailed Balance for MH
1. Write K(x'|x) = q(x'|x)α(x,x') for x'≠x
2. Show K(x'|x)p*(x) = min(q(x'|x)p*(x), q(x|x')p*(x')) = K(x|x')p*(x')
3. The min expression is symmetric in (x,x') ↔ swap ✓

### Recipe — Bayesian MH
1. Set p̄*(x) = p(y|x)p(x)
2. Log acceptance: log α = log p(y|x') + log p(x') + log q(x|x') − log p(y|x) − log p(x) − log q(x'|x)

### Common Exam Tasks
- Write out MH for a given target; compute acceptance ratio
- Prove MH satisfies detailed balance (Prop 4.2)
- Compare independent vs random walk proposals
- Design MH for Bayesian posterior with given prior/likelihood

## I2. Gibbs Sampling & Full Conditionals

**When**: Target has block structure; can sample each variable conditioned on the rest.

### Recipe — Deriving Full Conditionals
1. Write p*(x₁,…,x_d) ∝ f(x₁,…,x_d)
2. For variable x_m: treat all x_{-m} as constants
3. Collect terms involving x_m: p(x_m|x_{-m}) ∝ f(x₁,…,x_d) as function of x_m only
4. Recognise the kernel of a known distribution (Gaussian, Gamma, Beta, etc.)

### Recipe — Gibbs Sampler
1. Initialise (x₁⁰,…,x_d⁰)
2. For t=1,…,T: cycle through m=1,…,d:
   - x_m^t ~ p(x_m | x_1^t,…,x_{m-1}^t, x_{m+1}^{t-1},…,x_d^{t-1})
3. No accept/reject — every draw is accepted

### Example — Beta-Binomial Gibbs
- Model: P ~ Beta(α,β), X|P ~ Bin(n,P)
- Full conditionals: P|X ~ Beta(α+X, β+n−X), X|P ~ Bin(n,P)
- Cycle: sample P given X, then X given P

### Recipe — Metropolis-within-Gibbs
- When p(x_m|x_{-m}) is hard to sample:
  1. Instead of exact draw, run one MH step targeting p(x_m|x_{-m})
  2. Use any proposal q_m(x'_m|x_m)
  3. Accept/reject within that coordinate
  4. Continue cycling

### Recipe — Proving Gibbs Stationarity
1. Show each K_m satisfies detailed balance: K_m(x'|x)p*(x) = p_m*(x'_m|x_{-m})δ(x'_{-m}=x_{-m})p*(x) = p*(x'_m,x_{-m})δ(…) — symmetric
2. Each K_m leaves p* invariant ⟹ K = K₁∘…∘K_d does too

## J. Langevin Methods & MCMC Diagnostics

**When**: Target is differentiable; want gradient-based MCMC or need to assess chain quality.

### Recipe — ULA
1. Compute ∇log p*(x) (the score function)
2. Choose step size γ (small enough for stability)
3. Update: X_{n+1} = X_n + γ∇log p*(X_n) + √(2γ)V_n, V_n ~ N(0,I)
4. Note: biased — stationary distribution p^γ ≈ p* (exact only as γ→0)

### Recipe — MALA
1. Same proposal as ULA: q(x'|x) = N(x + γ∇log p*(x), 2γI)
2. Add MH accept/reject step with full acceptance ratio (q is NOT symmetric!)
3. Exact: targets p* for any γ (but small γ → high acceptance)

### Recipe — ULA Bias Analysis (Gaussian Target)
- If p* = N(μ, σ²): ULA stationary distribution is N(μ, 2σ⁴/(2σ²−γ))
- Bias in variance: σ²_ULA/σ² = σ²/(2σ²−γ) > 1
- Requires γ < 2σ² for stability

### Recipe — MCMC Diagnostics
1. **Trace plot**: plot samples x_t vs t; look for stationarity after burn-in
2. **Autocorrelation**: compute ρ_k = Cov(X_t,X_{t+k})/Var(X_t) for k=0,1,2,…
3. **ESS**: ESS = N/(1 + 2Σ_{k≥1} ρ_k); approximate by truncating sum when ρ_k ≈ 0
4. If ESS is low: chain mixes poorly → increase iterations, tune proposal, or use different method

### Recipe — Simulated Annealing
1. Define objective f(x) to minimise
2. Set p^{β_t}(x) ∝ exp(−β_t f(x)) with schedule β₁ < β₂ < … → ∞
3. Run MH targeting p^{β_t} at each step
4. Acceptance: α = min(1, exp(β_t(f(x_{t-1})−f(x'))) · q(x_{t-1}|x')/q(x'|x_{t-1}))
5. Early iterations (low β): broad exploration; later (high β): concentrate on minimum

## K. EM Algorithm & Parameter Learning

**When**: Learn parameters θ in latent variable models; MMLE problems.

### Recipe — EM Algorithm
1. Initialise θ₀
2. **E-step**: compute Q(θ, θ_k) = E_{p_{θ_k}(x|y)}[log p_θ(x,y)]
   - Often: expand log p_θ(x,y), take expectation w.r.t. posterior p_{θ_k}(x|y)
   - For Gaussian models: need E[X|y] and E[X²|y] (or Var[X|y])
3. **M-step**: θ_{k+1} = argmax_θ Q(θ, θ_k)
   - Differentiate Q w.r.t. θ, set to zero, solve

### Example — Gaussian EM (Example 5.2)
- Model: p_θ(x) = N(θ, σ₀²), p(y|x) = N(ax, σ²)
- E-step: p_{θ_k}(x|y) = N(m_k, s²) where m_k = (σ²θ_k + aσ₀²y)/(σ² + a²σ₀²)
- Q(θ, θ_k) = const − (θ−m_k)²/(2σ₀²)
- M-step: θ_{k+1} = m_k = rθ_k + b, where r = σ²/(σ²+a²σ₀²) < 1
- Converges: θ_k → y/a = θ* (same as MMLE)

### Recipe — Proving EM Ascent (Prop 5.1)
1. Write log p_θ(y) = log p_{θ_k}(y) + Δ(θ, θ_k) where Δ = Q(θ,θ_k) − Q(θ_k,θ_k)
2. Key inequality from Jensen's: log p_θ(y) ≥ log p_{θ_k}(y) + Δ(θ,θ_k)
3. Since θ_{k+1} maximises Q: Δ(θ_{k+1},θ_k) ≥ 0

### Recipe — Fisher's Identity for Gradient MMLE
1. ∇_θ log p_θ(y) = E_{p_θ(x|y)}[∇_θ log p_θ(x,y)]
2. Approximate with samples: ∇log p_θ(y) ≈ (1/K)Σ∇_θ log p_θ(x_k, y), x_k ~ p_θ(x|y)
3. Gradient ascent: θ_{t+1} = θ_t + δ · (approximate gradient)

### Recipe — PMMH
1. For current θ: estimate p̂(y|θ) via IS (or BPF for SSMs)
2. Propose θ' ~ q(·|θ)
3. Estimate p̂(y|θ') with fresh IS samples
4. Accept: α̂ = min(1, p̂(y|θ')p(θ')q(θ|θ') / (p̂(y|θ)p(θ)q(θ'|θ)))
5. Key: reuse p̂(y|θ) from previous iteration (don't re-estimate)

## L. Particle Filtering & SMC

**When**: Filtering in state-space models; sequential estimation with streaming data.

### Recipe — Bootstrap Particle Filter
1. **Initialise**: x₀^(i) ~ μ(x₀), i=1,…,N
2. For each time step t=1,2,…:
   - **Propagate**: x̃_t^(i) ~ f(x_t | x_{t-1}^(i))
   - **Weight**: W_t^(i) = g(y_t | x̃_t^(i)); normalise w̄_t^(i) = W_t^(i)/Σ W_t^(j)
   - **Estimate**: E[φ(X_t)|y_{1:t}] ≈ Σ w̄_t^(i)φ(x̃_t^(i))
   - **Resample**: draw x_t^(i) from {x̃_t^(j)} with probs w̄_t^(j)

### Recipe — Marginal Likelihood from BPF
1. At each step t: p̂(y_t|y_{1:t-1}) = (1/N)Σ_i g(y_t|x̃_t^(i)) = (1/N)Σ_i W_t^(i)
2. Full: p̂(y_{1:T}) = ∏_{t=1}^T p̂(y_t|y_{1:t-1})
3. Log-domain: log p̂(y_{1:T}) = Σ_t [logsumexp(log W_t^(1),…,log W_t^(N)) − log N]
4. This estimate is unbiased: E[p̂(y_{1:T})] = p(y_{1:T})

### Recipe — Particle MCMC
1. Choose prior p(θ), proposal q(θ'|θ)
2. At current θ: have p̂(y_{1:T}|θ) from previous BPF run
3. Propose θ' ~ q(·|θ)
4. Run BPF at θ' → get p̂(y_{1:T}|θ')
5. Accept: α = min(1, p̂(y|θ')p(θ')q(θ|θ') / (p̂(y|θ)p(θ)q(θ'|θ)))
6. Exact for p(θ|y_{1:T}) by pseudo-marginal argument

### Recipe — Deriving General PF Weights
1. Write unnormalised target: π̄_t = μ(x₀)∏f(x_k|x_{k-1})g(y_k|x_k)
2. Choose Markov proposal: q(x_{0:t}) = q(x₀)∏q(x_k|x_{k-1})
3. Weight ratio: W_{0:t} = π̄_t/q = W_{0:t-1} × [f(x_t|x_{t-1})g(y_t|x_t)/q(x_t|x_{t-1})]
4. For BPF: q = f → incremental weight = g(y_t|x_t)

### Common Exam Tasks
- Derive BPF weights for a given SSM
- Compute marginal likelihood estimate from particle weights
- Explain weight degeneracy in SIS and how resampling fixes it
- Describe evolutionary interpretation: mutation (propagate) + selection (weight + resample)
- Write particle MCMC acceptance ratio; explain why it's exact

## G. Proving Key Results

| Result | Proof Strategy |
|--------|---------------|
| Thm 2.1 (Prob. integral transform) | One line: P(F⁻¹(U)≤x) = P(U≤F(x)) = F(x) |
| Thm 2.2 (Fund. thm of simulation) | x-marginal of uniform on A: q(x) = ∫₀^{p̄*} (1/Z)dy = p̄*/Z = p* |
| Prop 2.3 (Acceptance rate) | â = E[a(X')] = ∫(p̄*/(Mq))q dx = Z/M |
| Box-Müller | 2D transformation of RVs formula; compute g⁻¹ and |det J| = 2 |
| Prop 2.2 (Marginalisation) | P(Z∈A) = ∫_{ℝ^{d₁}×A} p(x₁,x₂) dx₁dx₂ = ∫_A p_{X₂}(x₂)dx₂ |
| Curse of dimensionality | supM = K^d when p*=∏p₀, q=∏q₀; â = 1/K^d → 0 |
| Gaussian marginal likelihood | Complete the square and evaluate Gaussian integral |
| Conditional Bayes rule (Prop 1.1) | Apply standard Bayes rule with z fixed throughout |
| Prop 3.1 (MC unbiasedness) | E[φ̂_N] = (1/N)ΣE[φ(X^(i))] = E_{p*}[φ] by linearity |
| Prop 3.2 (MC variance) | Var(φ̂_N) = (1/N²)ΣVar(φ(X^(i))) = σ²/N by independence |
| Prop 3.5 (Optimal IS proposal) | Minimise Var_q[Wφ]; Lagrange multiplier or Cauchy-Schwarz → q* ∝ \|φ\|p* |
| Prop 4.1 (Detailed balance → invariance) | Integrate K(x'\|x)p*(x) = K(x\|x')p*(x') over x: ∫K(x'\|x)p*(x)dx = p*(x') ✓ |
| Prop 4.2 (MH detailed balance) | K(x'\|x)p*(x) = q(x'\|x)α(x,x')p*(x) = min(q(x'\|x)p*(x), q(x\|x')p*(x')); symmetric |
| Prop 4.3 (Gibbs stationarity) | Each K_m satisfies detailed balance by construction; compose → p* invariant |
| Prop 5.1 (EM ascent) | log p_θ(y) ≥ log p_{θ_k}(y) + Q(θ,θ_k)−Q(θ_k,θ_k); M-step ensures Δ≥0 |
| Prop 5.2 (Fisher's identity) | ∇log p_θ(y) = ∇p_θ(y)/p_θ(y) = ∫∇log p_θ(x,y)·p_θ(x\|y)dx |
| Prop 5.3 (PMMH correctness) | Extended target p̃*(θ,u) ∝ p(θ)m_θ(u)g(y,θ,u); m_θ cancels in ratio; θ-marginal = p(θ\|y) |

## M. Key Formulae Cheat Sheet

### Bayesian Inference
- Bayes: p(x|y) = p(y|x)p(x)/p(y)
- Gaussian (1D): μ_p = (σ²μ₀+σ₀²y)/(σ²+σ₀²), σ_p² = σ²σ₀²/(σ²+σ₀²)
- Gaussian (nD): Σ_p = (A^TΣ⁻¹A+Σ₀⁻¹)⁻¹, μ_p = Σ_p(A^TΣ⁻¹y+Σ₀⁻¹μ₀)
- Marginal likelihood (Gaussian): p(y) = N(y; μ₀, σ₀²+σ²)

### Transformation of RVs
- 1D: p_Y(y) = p_X(g⁻¹(y))|dg⁻¹/dy|
- nD: p_Y(y) = p_X(g⁻¹(y))|det J_{g⁻¹}|

### Sampling Algorithms
- Inverse transform: U ~ Unif, X = F⁻¹(U)
- Box-Müller: Z₁ = √(−2ln U₁)cos(2πU₂), Z₂ = √(−2ln U₁)sin(2πU₂)
- Multivariate Gaussian: Y = μ + Lv, Σ = LL^T
- Rejection: accept if U ≤ p̄*(X')/(Mq(X'))

### Rejection Sampling
- Optimal M: M* = sup_x p̄*(x)/q(x)
- Acceptance rate: â = 1/M (normalised), â = Z/M (unnormalised)
- Curse of dim.: â_d = π^{d/2}/Γ(d/2+1); for product: â = 1/K^d

### MC Integration & IS
- MC estimator: φ̂_N = (1/N)Σφ(X^(i)), Var = σ²/N
- IS weights: W^(i) = p̄*(X^(i))/q(X^(i))
- SNIS: w̄^(i) = W^(i)/Σ W^(j), φ̂ = Σ w̄^(i)φ(X^(i))
- Optimal IS: q* ∝ |φ|p*
- ESS (IS): (Σ W^(i))² / Σ(W^(i))²

### MCMC
- MH ratio: α = min(1, p̄*(x')q(x|x')/(p̄*(x)q(x'|x)))
- Random walk MH (symmetric q): α = min(1, p̄*(x')/p̄*(x))
- Gibbs: sample x_m ~ p(x_m|x_{-m}); no accept/reject
- ULA: X_{n+1} = X_n + γ∇log p* + √(2γ)V_n (biased)
- MALA: ULA proposal + MH correction (exact)
- SGLD: stochastic gradient + noise; gradient ≈ ∇log p + (n/K)Σ∇log p(y_{i_k}|x)
- ESS (MCMC): N/(1 + 2Σρ_k)
- Simulated annealing: p^β ∝ exp(−βf), α = min(1, exp(β(f(x)−f(x')))·q(x|x')/q(x'|x))

### Parameter Learning
- MMLE: θ* = argmax_θ log∫p_θ(x,y)dx
- EM: E-step Q(θ,θ_k) = E[log p_θ(x,y)], M-step θ_{k+1} = argmax Q
- Fisher's identity: ∇log p_θ(y) = E_{p_θ(x|y)}[∇log p_θ(x,y)]
- PMMH: α̂ = min(1, p̂(y|θ')p(θ')q(θ|θ')/(p̂(y|θ)p(θ)q(θ'|θ)))

### Sequential Monte Carlo
- SSM: X_0~μ, X_t|X_{t-1}~f, Y_t|X_t~g
- Joint: π̄_t = μ(x₀)∏f(x_k|x_{k-1})g(y_k|x_k)
- BPF weights: W_t^(i) = g(y_t|x̃_t^(i))
- BPF marginal likelihood: p̂(y_{1:t}) = ∏_k (1/N)Σ_i g(y_k|x̃_k^(i))
- General PF incremental: W_t = f(x_t|x_{t-1})g(y_t|x_t)/q(x_t|x_{t-1})
- Filtering: predict ξ_t = ∫f·π_{t-1}dx, update π_t = ξ_t·g/p(y_t|y_{1:t-1})

### Useful Distributions
- Exp(λ): p(x) = λe^{−λx}, F⁻¹(u) = −ln(1−u)/λ
- Gamma(α,β): p(x) ∝ x^{α−1}e^{−βx}
- Beta(α,β): p(x) ∝ x^{α−1}(1−x)^{β−1}
- Poisson(λ): P(k) = λ^k e^{−λ}/k!
- Weibull(k): F(x) = 1−exp(−x^k)

## N. Exam Strategy

### Likely Question Types
1. **Derive posterior** for given prior/likelihood (completing the square for Gaussian; recognise conjugate kernels)
2. **Derive inverse CDF** and write sampling algorithm (Thm 2.1)
3. **Prove transformation of RVs** (compute g⁻¹ and Jacobian; Box-Müller is a classic)
4. **Design rejection sampler**: find M, compute acceptance rate, optimise proposal
5. **Bayesian rejection sampling**: sample posterior without closed-form derivation
6. **Curse of dimensionality**: explain/prove why rejection fails in high-d
7. **Marginal likelihood** computation and model comparison
8. **IS/SNIS**: derive weights for given target/proposal; compute ESS; prove unbiasedness
9. **Design MH sampler**: write acceptance ratio, prove detailed balance
10. **Gibbs sampler**: derive full conditionals, write algorithm, prove stationarity
11. **Langevin methods**: write ULA/MALA update; analyse ULA bias for Gaussian target
12. **EM algorithm**: derive E-step and M-step; prove ascent property; show convergence
13. **Fisher's identity**: state and prove; use for gradient-based MMLE
14. **PMMH**: write algorithm; prove correctness via extended target; noise analysis
15. **BPF**: derive weights; compute marginal likelihood; explain evolutionary interpretation
16. **Particle MCMC**: write acceptance ratio; explain why unbiased ML estimate suffices
17. **Simulated annealing**: write algorithm; explain temperature schedule; acceptance ratio

### Common Mistakes to Avoid
- Forgetting absolute value in transformation formula (|dg⁻¹/dy| or |det J|)
- Using p* instead of p̄* (or vice versa) in rejection — be clear about normalised vs unnormalised
- Not checking that Mq(x) ≥ p̄*(x) for ALL x (including boundaries/tails)
- For disk sampling: forgetting the √r correction
- Confusing acceptance ratio a(x) with acceptance rate â = E[a(X')]
- IS: confusing unnormalised weights W with normalised weights w̄
- SNIS: forgetting it's biased (though consistent)
- MH: computing acceptance ratio with normalised instead of unnormalised target (both work, but be consistent)
- Gibbs: incorrect full conditional derivation — must treat x_{-m} as constants
- ULA: claiming it's exact (it's biased! Only MALA is exact)
- EM: not using posterior at θ_{t-1} (not θ) in E-step
- BPF: confusing propagation weights with cumulative weights

### Time Management
- Bayesian derivations are mechanical (expand, complete square) — do them quickly
- Rejection optimisation: differentiate log R, find x*, substitute — formulaic
- Proofs of Thm 2.1 and 2.2 are very short — write them out fully for easy marks
- MH detailed balance proof: the min-trick argument is ~3 lines
- EM ascent proof: ELBO argument is ~4-5 lines
- Fisher's identity proof: chain rule + multiply/divide by p_θ(y) is ~4 lines
- PMMH correctness: extended target + m cancellation is ~5 lines
- For BPF derivation: start from IS on path space, impose Markov proposal → recursive weights